# Exercise 3 — retrieve_and_answer

`retrieve_and_answer` is the single-turn RAG convenience function: search the retriever for the question, build a RAG prompt, call the LLM, and return `{question, docs, answer}`.  This is the fixed-pipeline baseline — the agent in Exercise 5 extends this with iteration.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

class SimpleRetriever:
    def __init__(self):
        self._docs = []
    def add(self, doc):
        self._docs.append(doc); return self
    def add_all(self, docs):
        for d in docs: self._docs.append(d)
        return self
    def _score(self, query, doc):
        q = set(query.lower().split())
        d = set(doc.content.lower().split())
        return len(q & d) / (len(q | d) + 1e-9)
    def search(self, query, top_k=3):
        if not self._docs: return []
        return sorted(self._docs, key=lambda doc: self._score(query, doc), reverse=True)[:top_k]
    def __len__(self): return len(self._docs)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
def format_docs(docs):
    if not docs:
        return "No documents found."
    lines = []
    for i, d in enumerate(docs, 1):
        source = d.metadata.get("source", f"doc{i}")
        lines.append(f"[{i}] ({source}) {d.content}")
    return "\n".join(lines)

def build_retrieval_prompt(question, docs):
    context = format_docs(docs)
    system = "\n".join([
        "You are a helpful assistant.",
        "Answer the question using ONLY the provided documents.",
        "If the answer is not in the documents, say: I don't have enough information.",
        "Cite document numbers like [1] when referencing specific facts.",
    ])
    user = "Documents:\n" + context + "\n\nQuestion: " + str(question)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

# ── Exercise: implement retrieve_and_answer ──────────────────────────────────

def retrieve_and_answer(question, retriever, top_k=3, llm_fn=None):
    # TODO:
    # 1. docs = retriever.search(question, top_k=top_k)
    # 2. prompt = build_retrieval_prompt(question, docs)
    # 3. answer = call_llm(prompt, llm_fn=llm_fn)
    # 4. return {"question": question, "docs": docs, "answer": answer}
    return {"question": question, "docs": [], "answer": ""}


### Checks

In [ ]:
checks = 0

retriever = SimpleRetriever()
retriever.add_all([
    Document("Python is a high-level programming language.", {"source": "intro"}),
    Document("Python was created by Guido van Rossum.", {"source": "history"}),
    Document("The sky is blue.", {"source": "nature"}),
])

mock_llm = lambda messages: "Python is a high-level programming language [1]."

# 1 — returns dict with question, docs, answer keys
try:
    r = retrieve_and_answer("What is Python?", retriever, llm_fn=mock_llm)
    assert "question" in r and "docs" in r and "answer" in r
    checks += 1; print("✅ 1 returns dict with question, docs, answer")
except Exception as e:
    print("❌ 1:", e)

# 2 — docs are retrieved (non-empty for relevant query)
try:
    r = retrieve_and_answer("What is Python?", retriever, llm_fn=mock_llm)
    assert len(r["docs"]) > 0
    checks += 1; print("✅ 2 docs are retrieved")
except Exception as e:
    print("❌ 2:", e)

# 3 — answer comes from llm_fn
try:
    r = retrieve_and_answer("What is Python?", retriever, llm_fn=mock_llm)
    assert r["answer"] == "Python is a high-level programming language [1]."
    checks += 1; print("✅ 3 answer is the llm_fn return value")
except Exception as e:
    print("❌ 3:", e)

# 4 — question is preserved
try:
    r = retrieve_and_answer("Who created Python?", retriever, llm_fn=mock_llm)
    assert r["question"] == "Who created Python?"
    checks += 1; print("✅ 4 question is preserved in result")
except Exception as e:
    print("❌ 4:", e)

# 5 — top_k limits results
try:
    r = retrieve_and_answer("Python", retriever, top_k=1, llm_fn=mock_llm)
    assert len(r["docs"]) <= 1
    checks += 1; print("✅ 5 top_k limits number of retrieved docs")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
